# 03 · Evaluation Protocol and Long-Tail Metrics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Souhaila-DJAFFAL/longtail-recsys/blob/main/notebooks/03_evaluation_metrics.ipynb)

How the models are measured, and the self-tests that check the measurement is correct.

**Protocol**
* **Full ranking.** For every user, *all* items are ranked; items already in the user's input history are
  excluded. Ranking against a small set of sampled negatives can change which model looks best
  (Krichene & Rendle, KDD 2020); with ~3.7k items, full ranking is cheap.
* **Pessimistic ties.** If the target ties with other items, it is placed *below* them, so a model cannot
  gain from constant scores.

**Accuracy** (one held-out item per user)

| Metric | Definition |
|---|---|
| HR@K | 1 if the target is in the top-K, else 0 |
| NDCG@K | $1/\log_2(\text{rank}+2)$ if the target is in the top-K (rank starts at 0), else 0 |
| NDCG@K head / tail | NDCG averaged separately over users whose target is a head item / a tail item |

**Popularity bias** (Abdollahpouri et al., 2019)

| Metric | Definition | Better |
|---|---|---|
| Coverage@K | Share of the catalogue that appears in at least one top-K list | higher |
| TailShare@K (APLT) | Average share of tail items in a top-K list | higher |
| ARP@K | Average training popularity of recommended items | lower |

**Popularity-aware re-ranking.** A simple post-processing mitigation that works with any model:

$$s'_{ui} = z_u(s_{ui}) - \lambda \, z(\log(1 + \text{pop}_i))$$

where $z_u$ standardizes the model's scores over candidate items for user $u$, and $\lambda \ge 0$
controls how strongly popular items are pushed down. $\lambda = 0$ is the unmodified model.

> Loaded by notebooks 04 and 05 with `%run`.

In [1]:
# --- Setup: works locally (Jupyter) and on Google Colab ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("02_models.ipynb"):
    !git clone -q https://github.com/Souhaila-DJAFFAL/longtail-recsys.git
    %cd longtail-recsys/notebooks
    !pip install -q tabulate

DATASET = os.environ.get("RECSYS_DATASET", "ml-1m")  # "ml-1m" (real) or "synthetic" (fast check)
QUICK = os.environ.get("RECSYS_QUICK", "0") == "1"     # few epochs, for smoke tests only
ARTIFACTS, RESULTS = "../artifacts", "../results"
os.makedirs(ARTIFACTS, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

/content/longtail-recsys/notebooks


In [2]:
import pickle
from types import SimpleNamespace
import numpy as np
import torch


def load_dataset(name=None):
    with open(f"{ARTIFACTS}/dataset_{name or DATASET}.pkl", "rb") as f:
        return SimpleNamespace(**pickle.load(f))


def pad_left(seqs, max_len):
    """List of item lists -> (n, max_len) array, most recent item last, 0 = padding."""
    out = np.zeros((len(seqs), max_len), dtype=np.int64)
    for r, s in enumerate(seqs):
        s = s[-max_len:]
        if len(s):
            out[r, -len(s):] = s
    return out


def head_items(item_pop, head_frac=0.2):
    """Boolean mask over item ids (index 0 = padding) marking the most popular head_frac of items."""
    n_items = len(item_pop) - 1
    order = np.argsort(-item_pop[1:], kind="stable") + 1
    is_head = np.zeros(len(item_pop), dtype=bool)
    is_head[order[: int(round(head_frac * n_items))]] = True
    return is_head


def popularity_penalty(item_pop):
    """Standardized log-popularity used for re-ranking."""
    lp = np.log1p(np.asarray(item_pop, dtype=np.float64))
    lp = (lp - lp[1:].mean()) / (lp[1:].std() + 1e-12)
    return torch.as_tensor(lp, dtype=torch.float32)

In [3]:
@torch.no_grad()
def evaluate(model, data, split="test", k=10, lambdas=(0.0,), head_frac=0.2, max_len=200,
             batch_size=512, device="cpu"):
    """Full-ranking evaluation on the validation or test target of every user.
    Returns one dict of metrics per re-ranking strength lambda."""
    model.eval()
    if split == "valid":
        hists, targets = data.train, data.valid
    elif split == "test":   # the validation item becomes part of the input history
        hists, targets = [s + [int(v)] for s, v in zip(data.train, data.valid)], data.test
    else:
        raise ValueError(split)

    is_head = head_items(data.item_pop, head_frac)
    is_head_t = torch.as_tensor(is_head, device=device)
    pop_t = torch.as_tensor(data.item_pop, dtype=torch.float32, device=device)
    pen = popularity_penalty(data.item_pop).to(device)
    acc = {lam: {"hit": [], "ndcg": [], "tail": [], "arp": [], "items": set()} for lam in lambdas}

    for start in range(0, data.n_users, batch_size):
        users = np.arange(start, min(start + batch_size, data.n_users))
        hist = torch.as_tensor(pad_left([hists[u] for u in users], max_len), device=device)
        raw = model.score(torch.as_tensor(users, device=device), hist).float()
        seen = torch.zeros_like(raw, dtype=torch.bool)
        seen[:, 0] = True
        for r, u in enumerate(users):
            seen[r, torch.as_tensor(hists[u], device=device)] = True
        tgt = torch.as_tensor(targets[users], device=device)

        # per-user standardization over candidate items
        n_cand = (~seen).sum(1, keepdim=True)
        mu = raw.masked_fill(seen, 0).sum(1, keepdim=True) / n_cand
        sd = (((raw - mu) ** 2).masked_fill(seen, 0).sum(1, keepdim=True) / n_cand).sqrt()
        z = (raw - mu) / (sd + 1e-12)

        for lam in lambdas:
            s = (z - lam * pen).masked_fill(seen, float("-inf"))
            rank = (s >= s.gather(1, tgt.unsqueeze(1))).sum(1) - 1   # pessimistic ties
            hit = rank < k
            top = s.topk(k, dim=1).indices
            a = acc[lam]
            a["hit"].append(hit.cpu().numpy())
            a["ndcg"].append(torch.where(hit, 1 / torch.log2(rank.float() + 2), 0.0).cpu().numpy())
            a["tail"].append((~is_head_t[top]).float().mean(1).cpu().numpy())
            a["arp"].append(pop_t[top].mean(1).cpu().numpy())
            a["items"].update(top.flatten().tolist())

    head_t = is_head[targets]
    out = []
    for lam in lambdas:
        a = acc[lam]
        hit, ndcg = np.concatenate(a["hit"]), np.concatenate(a["ndcg"])
        out.append({"lambda": lam, f"HR@{k}": hit.mean(), f"NDCG@{k}": ndcg.mean(),
                    f"NDCG@{k} head": ndcg[head_t].mean(),
                    f"NDCG@{k} tail": ndcg[~head_t].mean() if (~head_t).any() else float("nan"),
                    f"Coverage@{k}": len(a["items"]) / data.n_items,
                    f"TailShare@{k}": float(np.concatenate(a["tail"]).mean()),
                    f"ARP@{k}": float(np.concatenate(a["arp"]).mean())})
    return out

## Self-tests

Small, hand-built cases with known answers. They run every time this notebook is loaded, so a broken
metric stops the pipeline instead of silently producing wrong results.

In [4]:
def _toy_data(n_users=40, n_items=60, seed=0):
    rng = np.random.default_rng(seed)
    train, valid, test = [], [], []
    for _ in range(n_users):
        s = rng.choice(np.arange(1, n_items + 1), size=12, replace=False).tolist()
        train.append(s[:-2]); valid.append(s[-2]); test.append(s[-1])
    pop = np.zeros(n_items + 1, dtype=np.int64)
    for s in train:
        np.add.at(pop, s, 1)
    return SimpleNamespace(n_users=n_users, n_items=n_items, train=train, valid=np.array(valid),
                           test=np.array(test), item_pop=pop)


class _PointAt(torch.nn.Module):
    """Gives score 1 to one chosen item per user, 0 to everything else."""
    def __init__(self, items, n_items):
        super().__init__(); self.items, self.n = torch.as_tensor(items), n_items
    def score(self, users, hist):
        s = torch.zeros(len(users), self.n + 1)
        s[torch.arange(len(users)), self.items[users]] = 1.0
        return s


_d = _toy_data()
# 1. head = top 20% of items by popularity
assert head_items(np.array([0, 50, 40, 30, 20, 10, 5, 4, 3, 2, 1])).tolist() == [False, True, True] + [False] * 8
# 2. an oracle that points at the test item is perfect
_r = evaluate(_PointAt(_d.test, _d.n_items), _d, max_len=20)[0]
assert _r["HR@10"] == 1.0 and abs(_r["NDCG@10"] - 1.0) < 1e-6
# 3. pointing at an already-seen item never helps (seen items are excluded) and ties are pessimistic
_r = evaluate(_PointAt([s[0] for s in _d.train], _d.n_items), _d, max_len=20)[0]
assert _r["HR@10"] == 0.0
# 4. re-ranking with lambda > 0 moves tail items into the lists of a popularity-driven scorer
class _PopScorer(torch.nn.Module):
    def score(self, users, hist):
        return torch.as_tensor(_d.item_pop, dtype=torch.float32).repeat(len(users), 1)
_r0, _r1 = evaluate(_PopScorer(), _d, max_len=20, lambdas=(0.0, 2.0))
assert _r1["TailShare@10"] > _r0["TailShare@10"] and _r1["ARP@10"] < _r0["ARP@10"]
print("All evaluation self-tests passed.")

All evaluation self-tests passed.


**Next:** [`04_training.ipynb`](04_training.ipynb) trains BPR-MF and SASRec.